# 01 — Data Pipeline

Load, clean, and prepare raw data for analysis.

In [1]:
import sys
import os

# Adds to path
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Set plot style (try newer name first; fallback for older matplotlib)
# Reusable template
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
from src.api_client import fetch_all_stocks

# Load from cache — no API calls if CSVs already in data/raw/
stocks = fetch_all_stocks()

# Quick summary of what we have
for name, df in stocks.items():
    print(f"{name}: {len(df)} rows | {df.index.min().date()} to {df.index.max().date()}")

No data returned for IBM (IBM). Check symbol, API limit, or note field in response.
No data returned for Apple (AAPL). Check symbol, API limit, or note field in response.
No data returned for Microsoft (MSFT). Check symbol, API limit, or note field in response.
MTN Group: 251 rows | 2025-03-18 to 2026-03-18
Naspers: 251 rows | 2025-03-18 to 2026-03-18
Shoprite: 251 rows | 2025-03-18 to 2026-03-18
FirstRand: 251 rows | 2025-03-18 to 2026-03-18
Anglo American: 251 rows | 2025-03-18 to 2026-03-18


In [3]:
for name, df in stocks.items():
    print(f"\n{name}")
    print(f"  Missing values: {df.isnull().sum().sum()}")
    print(f"  Zero volume days: {(df['volume'] == 0).sum()}")
    print(f"  Duplicate dates: {df.index.duplicated().sum()}")
    print(df.describe().round(2))


MTN Group
  Missing values: 0
  Zero volume days: 0
  Duplicate dates: 0
           open      high       low     close       volume
count    251.00    251.00    251.00    251.00       251.00
mean   15202.35  15505.57  14981.70  15269.31   5647124.42
std     2527.11   2567.82   2469.14   2520.83   3395879.00
min    10375.00  10537.00   9952.00  10355.00    713054.00
25%    13164.50  13438.50  13005.00  13220.00   3511438.00
50%    15230.00  15580.00  15041.00  15357.00   5003176.00
75%    16990.00  17278.00  16774.00  17011.50   6565887.50
max    20801.00  21398.00  20403.00  20898.00  33684983.00

Naspers
  Missing values: 0
  Zero volume days: 0
  Duplicate dates: 0
            open       high        low      close       volume
count     251.00     251.00     251.00     251.00       251.00
mean   107869.33  109208.74  106657.74  107788.34   1995603.14
std     11245.50   11229.89   11295.89   11213.62   1263665.02
min     79953.20   86768.00   79643.00   84438.00    359481.00
25%    1

In [4]:
# Daily return = percentage change in closing price day to day.
# Using returns (not raw price) lets you compare stocks that trade at different price levels.
returns = pd.DataFrame()

for name, df in stocks.items():
    returns[name] = df['close'].pct_change() * 100  # as percentage

returns.dropna(inplace=True)

print("Daily returns (%) — last 5 days:")
print(returns.tail().round(3))
print("\nAverage daily return (%):")
print(returns.mean().round(3))
print("\nVolatility (std dev of daily returns, %):")
print(returns.std().round(3))

Daily returns (%) — last 5 days:
            MTN Group  Naspers  Shoprite  FirstRand  Anglo American
Date                                                               
2026-03-12     -1.415    0.655    -1.460     -2.406          -1.782
2026-03-13     -0.571    3.424     0.012     -0.011          -1.852
2026-03-16      7.003    0.497     4.139      1.307          -0.867
2026-03-17     -0.041    0.505    -0.773      1.144           0.451
2026-03-18      1.145   -6.987    -1.370     -1.752          -2.065

Average daily return (%):
MTN Group         0.244
Naspers          -0.005
Shoprite          0.009
FirstRand         0.086
Anglo American    0.120
dtype: float64

Volatility (std dev of daily returns, %):
MTN Group         2.298
Naspers           1.844
Shoprite          1.468
FirstRand         1.699
Anglo American    2.278
dtype: float64
